In [1]:
import yfinance as yf
import pandas as pd

ticker = "EURUSD=X"
start_date = "2018-01-01"
end_date = "2024-01-01"

def download_stock_data(ticker, start_date, end_date):
    df = yf.download(ticker, start=start_date, end=end_date, interval="1d")
    df.columns = df.columns.get_level_values(0)  # flatten here
    return df

df = download_stock_data(ticker, start_date, end_date)

df.to_csv("eurusd_data.csv")

[*********************100%***********************]  1 of 1 completed


In [2]:
print(df.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


In [3]:
print(df.dtypes)

Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object


In [4]:
df['High'] = pd.to_numeric(df['High'], errors='coerce')
df['Low'] = pd.to_numeric(df['Low'], errors='coerce')
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
df['Open'] = pd.to_numeric(df['Open'], errors='coerce')

def calculate_atr(df, period=14):
    high = df['High'].squeeze()
    low = df['Low'].squeeze()
    close = df['Close'].squeeze()
    
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    
    tr = pd.DataFrame({'tr1': tr1, 'tr2': tr2, 'tr3': tr3}).max(axis=1)
    
    df['ATR'] = tr.rolling(window=14).mean()
    return df

df = calculate_atr(df)
print(df['ATR'].tail(10))

Date
2023-12-18    0.008939
2023-12-19    0.009184
2023-12-20    0.008790
2023-12-21    0.008228
2023-12-22    0.008289
2023-12-25    0.007839
2023-12-26    0.007595
2023-12-27    0.008045
2023-12-28    0.008241
2023-12-29    0.008358
Name: ATR, dtype: float64


In [5]:
print(df['ATR'].describe())
print(df['ATR'].mean())

count    1551.000000
mean        0.008611
std         0.002567
min         0.003631
25%         0.006890
50%         0.008287
75%         0.009945
max         0.021309
Name: ATR, dtype: float64
0.008611474235248663


In [6]:
from scipy.signal import argrelextrema
import numpy as np

def find_sr_levels(df, order=10, tolerance=0.02):
    close = df['Close'].squeeze().values
    
    # find pivot highs and lows
    pivot_highs = argrelextrema(close, np.greater, order=order)[0]
    pivot_lows = argrelextrema(close, np.less, order=order)[0]
    
    resistance_levels = close[pivot_highs]
    support_levels = close[pivot_lows]
    
    return support_levels, resistance_levels

support, resistance = find_sr_levels(df)

print(f"Support levels found: {len(support)}")
print(f"Resistance levels found: {len(resistance)}")
print(f"Sample supports: {support[:5]}")
print(f"Sample resistances: {resistance[:5]}")

Support levels found: 57
Resistance levels found: 50
Sample supports: [1.19293308 1.22527993 1.21912563 1.22518992 1.22515988]
Sample resistances: [1.25100088 1.24166536 1.24574888 1.23851275 1.18077695]


In [7]:
def is_near_support(price, support_levels, tolerance=0.02):
    for level in support_levels:
        if abs(price - level) / level < tolerance:  # within 2%
            return True
    return False

def is_near_resistance(price, resistance_levels, tolerance=0.02):
    for level in resistance_levels:
        if abs(price - level) / level < tolerance:  # within 2%
            return True
    return False

In [8]:
def detect_double_bottom(df, support_levels, order=1, tolerance=0.08):
    close = df['Close'].values
    low = df['Low'].values
    
    pivot_lows = argrelextrema(close, np.less, order=order)[0]
    
    signals = []
    
    for i in range(1, len(pivot_lows)):
        idx1 = pivot_lows[i-1]
        idx2 = pivot_lows[i]
        
        low1 = low[idx1]
        low2 = low[idx2]
        
        # condition 1 — two lows within 3% of each other
        if abs(low1 - low2) / low1 > tolerance:
            continue
            
        # condition 2 — peak in between
        peak_between = close[idx1:idx2].max()
        if peak_between < low1 * 0.8:
            continue
        
        # condition 3 — near support
        if not is_near_support(low2, support_levels, tolerance):
            continue
        
        # condition 4 — neckline breakout within 5 candles
        neckline = peak_between
        breakout_idx = None
        for j in range(1, 3):
            if idx2 + j < len(close):
                if close[idx2 + j] > neckline:
                    breakout_idx = idx2 + j
                    break
        
        if breakout_idx is None:
            continue
        
        signals.append({
            'date': df.index[breakout_idx],
            'entry': close[breakout_idx],
            'low1': low1,
            'low2': low2,
            'neckline': neckline
        })
    
    return signals

signals = detect_double_bottom(df, support)
print(f"Signals found: {len(signals)}")
for s in signals[:9]:
    print(s)

Signals found: 147
{'date': Timestamp('2018-03-22 00:00:00'), 'entry': np.float64(1.2347661256790161), 'low1': np.float64(1.226046085357666), 'low2': np.float64(1.2246992588043213), 'neckline': np.float64(1.2336845397949219)}
{'date': Timestamp('2018-03-26 00:00:00'), 'entry': np.float64(1.235437273979187), 'low1': np.float64(1.2246992588043213), 'low2': np.float64(1.2319974899291992), 'neckline': np.float64(1.2347661256790161)}
{'date': Timestamp('2018-04-10 00:00:00'), 'entry': np.float64(1.2322403192520142), 'low1': np.float64(1.225760579109192), 'low2': np.float64(1.2218068838119507), 'neckline': np.float64(1.228682279586792)}
{'date': Timestamp('2018-04-17 00:00:00'), 'entry': np.float64(1.2385127544403076), 'low1': np.float64(1.2218068838119507), 'low2': np.float64(1.2308752536773682), 'neckline': np.float64(1.2371643781661987)}
{'date': Timestamp('2018-06-05 00:00:00'), 'entry': np.float64(1.1700559854507446), 'low1': np.float64(1.1520073413848877), 'low2': np.float64(1.16687476

In [9]:
def calculate_trade_levels(signals, df):
    trades = []
    
    for signal in signals:
        entry = signal['entry']
        low2 = signal['low2']
        date = signal['date']
        
        # get ATR at entry date
        atr = df.loc[date, 'ATR']
        
        # stop = second bottom - ATR
        stop = low2 - atr
        
        # risk = entry - stop
        risk = entry - stop
        
        # target = entry + 2.5 * risk
        target = entry + 2.5 * risk
        
        trades.append({
            'date': date,
            'entry': entry,
            'stop': stop,
            'target': target,
            'risk': risk,
            'reward': target - entry,
            'rr_ratio': (target - entry) / risk
        })
    
    return trades

trades = calculate_trade_levels(signals, df)

# check first 3 trades
for t in trades[:3]:
    print(f"Date: {t['date'].date()}")
    print(f"Entry: {t['entry']:.5f}")
    print(f"Stop: {t['stop']:.5f}")
    print(f"Target: {t['target']:.5f}")
    print(f"R:R = {t['rr_ratio']:.2f}")
    print()

Date: 2018-03-22
Entry: 1.23477
Stop: 1.21504
Target: 1.28409
R:R = 2.50

Date: 2018-03-26
Entry: 1.23544
Stop: 1.22246
Target: 1.26788
R:R = 2.50

Date: 2018-04-10
Entry: 1.23224
Stop: 1.21307
Target: 1.28018
R:R = 2.50



In [10]:
def backtest_trades(trades, df):
    results = []
    
    for trade in trades:
        if np.isnan(trade['stop']):
            continue
            
        entry_date = trade['date']
        entry_idx = df.index.get_loc(entry_date)
        
        # check next 50 candles
        for i in range(1, 50):
            if entry_idx + i >= len(df):
                break
            
            high = df['High'].iloc[entry_idx + i]
            low = df['Low'].iloc[entry_idx + i]
            
            # hit target first = WIN
            if high >= trade['target']:
                results.append({
                    'entry_date': entry_date,
                    'exit_date': df.index[entry_idx + i],
                    'outcome': 'WIN',
                    'pnl': trade['reward']
                })
                break
            
            # hit stop first = LOSS
            if low <= trade['stop']:
                results.append({
                    'entry_date': entry_date,
                    'exit_date': df.index[entry_idx + i],
                    'outcome': 'LOSS',
                    'pnl': -trade['risk']
                })
                break
    
    return results

results = backtest_trades(trades, df)
print(f"Total trades: {len(results)}")

Total trades: 125


In [11]:
wins = [r for r in results if r['outcome'] == 'WIN']
losses = [r for r in results if r['outcome'] == 'LOSS']

win_rate = len(wins) / len(results) * 100
total_pnl = sum([r['pnl'] for r in results])
avg_win = np.mean([r['pnl'] for r in wins]) if wins else 0
avg_loss = np.mean([r['pnl'] for r in losses]) if losses else 0

print(f"Win Rate: {win_rate:.2f}%")
print(f"Wins: {len(wins)}")
print(f"Losses: {len(losses)}")
print(f"Total P&L: {total_pnl:.5f}")
print(f"Average Win: {avg_win:.5f}")
print(f"Average Loss: {avg_loss:.5f}")
print(f"Profit Factor: {abs(sum([r['pnl'] for r in wins]) / sum([r['pnl'] for r in losses])):.2f}" if losses else "N/A")

Win Rate: 20.80%
Wins: 26
Losses: 99
Total P&L: -0.42319
Average Win: 0.04185
Average Loss: -0.01526
Profit Factor: 0.72
